# 04 · The compliance view (slide-16 payoff)

Every gate run you executed in notebooks 02–03 landed an immutable, timestamped record in MLflow: the verdict, per-benchmark scores, model identity, collection id, and owner — one record, the same one engineering, ML, and compliance all read.

With docker down, the records live in a local SQLite store (`mlruns.db`). This notebook reads them straight from there.

In [ ]:
import os
from pathlib import Path

# Notebooks live in <repo>/notebooks; walk up to the dir that holds the Makefile.
root = Path.cwd()
while not (root / "Makefile").exists() and root != root.parent:
    root = root.parent
os.chdir(root)
print("working dir:", Path.cwd())

## Read the records from local MLflow
Filtered to `tags.pipeline = 'release-gate'` — the gate + agent runs.

In [ ]:
import mlflow
from pathlib import Path

db = Path.cwd() / "mlruns.db"
if not db.exists():
    print("No mlruns.db yet — run 02-release-gate-blocked.ipynb first.")
else:
    mlflow.set_tracking_uri(f"sqlite:///{db}")
    df = mlflow.search_runs(
        experiment_names=["edd-release-gate"],
        filter_string="tags.pipeline = 'release-gate'",
    )
    cols = [c for c in [
        "tags.mlflow.runName", "tags.candidate", "tags.component",
        "tags.step.name", "tags.step.status", "start_time",
    ] if c in df.columns]
    print(f"{len(df)} runs\n")
    display(df[cols] if cols else df)

## The interactive MLflow UI
For the full experience — drill into a run, see the nested per-step trace, read the verdict on one timestamped record — launch the MLflow UI against the same store:

```bash
mlflow ui --backend-store-uri sqlite:///mlruns.db
```

Then open <http://localhost:5000>, pick experiment **`edd-release-gate`**, and filter `tags.pipeline = 'release-gate'`.

Run it from a terminal (it's a long-running server). The cell below is commented so it doesn't block the kernel:

In [ ]:
# !mlflow ui --backend-store-uri sqlite:///mlruns.db   # then open http://localhost:5000

## Final fallback — the static compliance view
If even MLflow won't cooperate on stage, `mock-mlflow-compliance-view.html` at the repo root **is** the compliance view (with a BEFORE/AFTER toggle). Open it in any browser — no kernel, no server, no network.

In [ ]:
print("Open in a browser:", Path.cwd() / "mock-mlflow-compliance-view.html")